# Exploración de los datos de RRHH
Revisando e identificando valores nulos, identificadores duplicados, fechas con formatos incorrectos y problemas en salarios, correos y categorías. Los resultados permiten comprender las reglas de validación utilizadas en el pipeline.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
base = Path.cwd()
con = sqlite3.connect((base / 'datos' / 'rrhh.sqlite').resolve().as_uri() + '?mode=ro', uri=True)
df = pd.read_sql_query('SELECT * FROM empleados', con)
con.close()
print('Filas:', len(df))
print(df.head().to_string(index=False))

Filas: 2575
id_empleado    nombre apellido                         email fecha_ingreso departamento             cargo   salario  modalidad
    EMP0001  Agustina   Medina agustina.medina1@tecnosur.com    25/04/2019   Tecnologia   Soporte tecnico 1175000.0    hibrido
    EMP0002    Franco   Acosta   franco.acosta2@tecnosur.com    2023-02-12     Finanzas          Contador 1675000.0     remoto
    EMP0003 Valentina     Ruiz  valentina.ruiz3@tecnosur.com    2017-05-27   Tecnologia Analista de datos 2050000.0 presencial
    EMP0004    Franco    Ramos    franco.ramos4@tecnosur.com    2019-01-02         None          Contador 1700000.0 presencial
    EMP0005    Martin   Medina   martin.medina5@tecnosur.com    2015-05-07   Tecnologia   Soporte tecnico 1400000.0 presencial


In [2]:
print(df.dtypes.to_string())
print('Nulos por columna:')
print(df.isna().sum().to_string())
print('Duplicados de ID:', df.duplicated('id_empleado').sum())

id_empleado       object
nombre            object
apellido          object
email             object
fecha_ingreso     object
departamento      object
cargo             object
salario          float64
modalidad         object
Nulos por columna:
id_empleado        0
nombre             0
apellido           0
email            153
fecha_ingreso      0
departamento     160
cargo              0
salario            0
modalidad          0
Duplicados de ID: 75


In [3]:
for columna in ['departamento', 'modalidad', 'cargo']:
    print(columna)
    print(df[columna].value_counts(dropna=False).to_string())

departamento
departamento
Tecnologia    793
RRHH          552
Finanzas      547
Comercial     523
None          160
modalidad
modalidad
remoto        913
presencial    856
hibrido       806
cargo
cargo
Tesorero                311
Generalista             304
Ejecutivo de cuentas    296
Reclutador              289
Desarrollador           285
Analista de datos       284
Soporte tecnico         279
Contador                266
Vendedor                261


In [4]:
fechas = pd.to_datetime(df['fecha_ingreso'], format='%Y-%m-%d', errors='coerce')
print('Fechas fuera de formato ISO:', fechas.isna().sum())
print(df.loc[fechas.isna(), 'fecha_ingreso'].head(10).to_string(index=False))
print('Salarios no positivos:', (df['salario'] <= 0).sum())
print('Emails sin formato básico:', (~df['email'].fillna('').str.contains(r'.+@.+\..+', regex=True)).sum())

Fechas fuera de formato ISO: 256
25/04/2019
09/05/2023
08/09/2022
21/06/2021
09/12/2017
22/02/2018
07/12/2021
22/05/2023
29/08/2018
12/03/2018
Salarios no positivos: 102
Emails sin formato básico: 153


## Decisiones de limpieza
Se convierten las fechas al formato año-mes-día, se completan los departamentos según el cargo, se transforman los salarios negativos en positivos y se reconstruyen los correos inválidos utilizando nombre, apellido e identificador. Se descartan los registros duplicados posteriores al primero. La validación registra el primer motivo de rechazo detectado en cada fila.